# 07. BigQuery Upload

## 목표
- `data/tableau/` 폴더의 CSV 6개를 BigQuery 테이블로 업로드
- ADC(Application Default Credentials) 방식으로 인증 (키 파일 없음)

## 업로드 대상
| 파일명 | 테이블명 |
|---|---|
| aarrr_funnel.csv | aarrr_funnel |
| yearly_trend.csv | yearly_trend |
| cohort_behavior.csv | cohort_behavior |
| rfm_segments.csv | rfm_segments |
| ab_test_result.csv | ab_test_result |
| mcc_category.csv | mcc_category |

In [1]:
import pandas as pd
from google.cloud import bigquery
from pathlib import Path

# 프로젝트 설정
PROJECT_ID = "project-08140dda-e851-4d93-b88"
DATASET_ID = "ibm_card_analysis"
DATA_DIR = Path("../data/tableau")

# BigQuery 클라이언트 초기화 (ADC 자동 참조)
client = bigquery.Client(project=PROJECT_ID)
print(f"인증 완료: {client.project}")

인증 완료: project-08140dda-e851-4d93-b88


## 업로드 실행
- `write_disposition`: WRITE_TRUNCATE → 테이블이 이미 있으면 덮어씀
- `autodetect`: True → 스키마 자동 감지

In [2]:
def upload_csv_to_bq(file_name: str, table_name: str):
    file_path = DATA_DIR / file_name
    df = pd.read_csv(file_path)
    
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    
    job_config = bigquery.LoadJobConfig(
        autodetect=True,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )
    
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()  # 완료까지 대기
    
    table = client.get_table(table_id)
    print(f"[완료] {table_name} → {table.num_rows}행 {len(table.schema)}열 업로드")

# 업로드 대상 목록
targets = [
    ("aarrr_funnel.csv",    "aarrr_funnel"),
    ("yearly_trend.csv",    "yearly_trend"),
    ("cohort_behavior.csv", "cohort_behavior"),
    ("rfm_segments.csv",    "rfm_segments"),
    ("ab_test_result.csv",  "ab_test_result"),
    ("mcc_category.csv",    "mcc_category"),
]

for file_name, table_name in targets:
    upload_csv_to_bq(file_name, table_name)

c:\Users\seonu\anaconda3\envs\ibm-card-analysis\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[완료] aarrr_funnel → 3행 3열 업로드


c:\Users\seonu\anaconda3\envs\ibm-card-analysis\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[완료] yearly_trend → 29행 5열 업로드


c:\Users\seonu\anaconda3\envs\ibm-card-analysis\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[완료] cohort_behavior → 153행 5열 업로드


c:\Users\seonu\anaconda3\envs\ibm-card-analysis\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
c:\Users\seonu\anaconda3\envs\ibm-card-analysis\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[완료] rfm_segments → 1337행 9열 업로드
[완료] ab_test_result → 3행 4열 업로드


c:\Users\seonu\anaconda3\envs\ibm-card-analysis\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[완료] mcc_category → 20행 5열 업로드


## 업로드 결과 검증
BigQuery에 실제로 올라갔는지 테이블 목록으로 확인

In [3]:
tables = client.list_tables(f"{PROJECT_ID}.{DATASET_ID}")
print(f"=== {DATASET_ID} 데이터셋 테이블 목록 ===")
for table in tables:
    print(f" - {table.table_id}")

=== ibm_card_analysis 데이터셋 테이블 목록 ===
 - aarrr_funnel
 - ab_test_result
 - cohort_behavior
 - mcc_category
 - rfm_segments
 - yearly_trend


## 원본 데이터 BigQuery 업로드 (GCS 경유)

2,400만 행 대용량 파일을 메모리 문제 없이 올리기 위해
GCS(Google Cloud Storage) 경유 방식을 사용한다.

### 흐름
로컬 CSV → GCS 버킷 업로드 → BigQuery가 GCS에서 직접 로드

### 업로드 대상
| 파일명 | 테이블명 |
|---|---|
| credit_card_transactions-ibm_v2.csv | transactions |
| sd254_cards.csv | cards |
| sd254_users.csv | users |

In [4]:
from google.cloud import storage

# GCS 설정
BUCKET_NAME = "ibm-card-raw-data"
RAW_DIR = Path("../data/raw")

# GCS 클라이언트 초기화
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

print(f"GCS 버킷 연결 완료: {BUCKET_NAME}")

GCS 버킷 연결 완료: ibm-card-raw-data


### Step 1. 로컬 CSV → GCS 업로드
- 2.18GB 파일은 업로드에 수 분 소요될 수 있음
- 진행 상황을 실시간으로 출력

In [5]:
import os

def upload_to_gcs(file_name: str):
    file_path = RAW_DIR / file_name
    blob = bucket.blob(file_name)
    
    file_size = os.path.getsize(file_path) / (1024 ** 2)  # MB
    print(f"업로드 시작: {file_name} ({file_size:.1f} MB)")
    
    blob.upload_from_filename(str(file_path))
    print(f"[완료] GCS 업로드: gs://{BUCKET_NAME}/{file_name}\n")

raw_files = [
    "credit_card_transactions-ibm_v2.csv",
    "sd254_cards.csv",
    "sd254_users.csv",
]

for file_name in raw_files:
    upload_to_gcs(file_name)

업로드 시작: credit_card_transactions-ibm_v2.csv (2241.8 MB)
[완료] GCS 업로드: gs://ibm-card-raw-data/credit_card_transactions-ibm_v2.csv

업로드 시작: sd254_cards.csv (0.5 MB)
[완료] GCS 업로드: gs://ibm-card-raw-data/sd254_cards.csv

업로드 시작: sd254_users.csv (0.2 MB)
[완료] GCS 업로드: gs://ibm-card-raw-data/sd254_users.csv



### Step 2. GCS → BigQuery 로드
- pandas 메모리 사용 없이 BigQuery가 GCS에서 직접 읽어서 로드
- 스키마 자동 감지

In [8]:
def load_gcs_to_bq(file_name: str, table_name: str):
    uri = f"gs://{BUCKET_NAME}/{file_name}"
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    
    job_config = bigquery.LoadJobConfig(
    autodetect=True,
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    column_name_character_map="V2",
)
    
    print(f"BigQuery 로드 시작: {uri}")
    job = client.load_table_from_uri(uri, table_id, job_config=job_config)
    job.result()  # 완료까지 대기
    
    table = client.get_table(table_id)
    print(f"[완료] {table_name} → {table.num_rows:,}행 {len(table.schema)}열 로드\n")

targets = [
    ("credit_card_transactions-ibm_v2.csv", "transactions"),
    ("sd254_cards.csv",                     "cards"),
    ("sd254_users.csv",                     "users"),
]

for file_name, table_name in targets:
    load_gcs_to_bq(file_name, table_name)

BigQuery 로드 시작: gs://ibm-card-raw-data/credit_card_transactions-ibm_v2.csv
[완료] transactions → 24,386,900행 15열 로드

BigQuery 로드 시작: gs://ibm-card-raw-data/sd254_cards.csv
[완료] cards → 6,146행 13열 로드

BigQuery 로드 시작: gs://ibm-card-raw-data/sd254_users.csv
[완료] users → 2,000행 18열 로드



### Step 3. 업로드 결과 검증

In [9]:
tables = client.list_tables(f"{PROJECT_ID}.{DATASET_ID}")
print(f"=== {DATASET_ID} 데이터셋 전체 테이블 목록 ===")
for table in tables:
    t = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{table.table_id}")
    print(f" - {table.table_id}: {t.num_rows:,}행")

=== ibm_card_analysis 데이터셋 전체 테이블 목록 ===
 - aarrr_funnel: 3행
 - ab_test_result: 3행
 - cards: 6,146행
 - cohort_behavior: 153행
 - mcc_category: 20행
 - rfm_segments: 1,337행
 - transactions: 24,386,900행
 - users: 2,000행
 - yearly_trend: 29행
